# Deepfake Audio Detection

This notebook is a thin, reproducible walkthrough for the project code. The main implementation lives in `src/`, `scripts/train.py`, `predict.py`, and `app.py` so the same pipeline works from notebook, CLI, and Streamlit.

## Workflow

1. Read WAV files directly from `for-2sec.zip`.
2. Normalize audio to mono, 16 kHz, 2 seconds.
3. Extract MFCC, delta MFCC, log-mel, temporal pooling, and spectral statistics.
4. Train a gradient boosting classifier.
5. Report accuracy, EER, macro F1, per-class accuracy, and confusion matrix.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from deepfake_audio_detector.dataset import find_dataset_zip, list_examples

dataset_zip = find_dataset_zip(ROOT)
examples = list_examples(dataset_zip)
dataset_zip, len(examples)

In [ ]:
import pandas as pd

pd.DataFrame([e.__dict__ for e in examples]).groupby(['split', 'label_name']).size()

## Train

Run the full benchmark training command from a terminal:

```powershell
.\.venv\Scripts\python.exe scripts\train.py --include-validation-in-training --operating-threshold 0.02299546758094686
```

For a quick notebook smoke test, run a small version below.

In [ ]:
# Optional smoke test. It overwrites reports/model with a tiny model, so keep it commented for final runs.
# !.\.venv\Scripts\python.exe scripts\train.py --limit-per-class 20 --force-features --workers 4

In [ ]:
import json

metrics_path = ROOT / 'reports' / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    pd.DataFrame([
        {
            'split': split,
            'accuracy': report['accuracy'],
            'eer': report['eer'],
            'macro_f1': report['f1_macro'],
            'real_accuracy': report['per_class_accuracy']['real'],
            'fake_accuracy': report['per_class_accuracy']['fake'],
        }
        for split, report in metrics.items()
    ])
else:
    print('Run scripts/train.py first.')